# 06 - Évaluation & Prédiction de Prix

Métriques comparatives, analyse des résidus et simulation pour un nouveau logement.

**Ce notebook répond à la question finale :**
'Mon modèle est-il bon ? Et quel prix proposer pour mon appartement ?'

## Chargement et ré-entraînement

Chaque notebook est **autonome** : on ré-entraîne ici pour ne pas dépendre
de variables définies dans le notebook 05 (les notebooks ne partagent pas leur mémoire).

On stocke tout dans `models[ville][modele]` : dictionnaire imbriqué.

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

CITIES = {'Lyon':'lyon','Paris':'paris','Bordeaux':'bordeaux'}
COLORS = {'Lyon':'steelblue','Paris':'coral','Bordeaux':'mediumseagreen'}
TARGET = 'price'
SIMPLE_FEAT = ['accommodates']
MULTIPLE_FEAT = [
    'accommodates','bedrooms','beds','bathrooms','minimum_nights',
    'availability_365','number_of_reviews','reviews_per_month',
    'review_scores_rating','review_scores_cleanliness','review_scores_location',
    'host_is_superhost','host_response_rate','calculated_host_listings_count',
    'instant_bookable','room_type_code','neighbourhood_freq','property_type_freq',
]

dfs = {n: pd.read_csv(f'../data/processed/{k}_features.csv') for n, k in CITIES.items()}

def train(df, features):
    """Entraîne un LinearRegression, retourne (modèle, X_test, y_test, y_pred, features)."""
    feats = [f for f in features if f in df.columns]
    d = df[feats + [TARGET]].dropna()
    X, y = d[feats], d[TARGET]
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=42)
    m = LinearRegression().fit(Xtr, ytr)
    yp = m.predict(Xte)
    return m, Xte, yte, yp, feats

models = {n: {'simple': train(df,SIMPLE_FEAT), 'multiple': train(df,MULTIPLE_FEAT)}
          for n, df in dfs.items()}
print('Modèles entraînés.')

## Tableau comparatif des métriques

**MAE** (Mean Absolute Error) : erreur moyenne en euros.
→ MAE = 33€ : le modèle se trompe en moyenne de 33€.

**RMSE** (Root Mean Squared Error) : comme MAE mais pénalise les grosses erreurs
(l'écart est mis au carré avant la moyenne). RMSE > MAE signifie quelques grosses erreurs.

**R²** (coefficient de détermination) : proportion de variance expliquée.
- R² = 0 : pas mieux que prédire toujours la moyenne
- R² = 1 : prédiction parfaite
- R² = 0.38 : le modèle explique 38% de la variabilité des prix

**Pourquoi R² modéré ?** Le prix dépend aussi de : photos, description, saison, mobilier...

In [ ]:
rows = []
for name in CITIES:
    for label in ['simple','multiple']:
        m, Xte, yte, yp, _ = models[name][label]
        rows.append({
            'Ville': name, 'Modèle': label.capitalize(),
            'MAE (€)' : round(mean_absolute_error(yte,yp),2),
            'RMSE (€)': round(np.sqrt(mean_squared_error(yte,yp)),2),
            'R²'      : round(r2_score(yte,yp),4),
            'Nb test' : len(yte),
        })
display(pd.DataFrame(rows))

## Analyse des résidus

**Résidu = prix réel − prix prédit**

**Graphique idéal :** points répartis aléatoirement autour de 0 (bande horizontale).
→ le modèle se trompe de façon aléatoire, pas systématiquement.

**Si on voit une courbe ou un entonnoir :** il reste une structure que le modèle n'a pas capturée
→ la relation n'est pas parfaitement linéaire (attendu avec des prix AirBnB).

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(14, 14))
for row, name in enumerate(CITIES):
    for col, label in enumerate(['simple','multiple']):
        m, Xte, yte, yp, _ = models[name][label]
        res = yte.values - yp  # résidus
        ax  = axes[row][col]
        ax.scatter(yp, res, alpha=0.25, s=8, color=COLORS[name])
        ax.axhline(0, color='black', linewidth=1, linestyle='--')  # ligne à 0
        ax.set_title(f'Résidus - {name} ({label})')
        ax.set_xlabel('Prix prédit (€)'); ax.set_ylabel('Résidu (€)')
plt.tight_layout()
plt.savefig('../reports/figures/residus.png', bbox_inches='tight')
plt.show()

## Prédiction pour un nouveau logement

**Cas concret :** T2, 4 personnes, note 4.8/5, superhost, appartement entier, centre-ville.

`pd.DataFrame([[...]], columns=feats)` : DataFrame d'**une seule ligne** dans l'ordre exact
des features utilisées à l'entraînement — scikit-learn exige cet ordre.

`model.predict(X_new)[0]` : retourne un tableau numpy → `float()` le convertit en nombre Python.

In [ ]:
new_listing = dict(
    accommodates=4, bedrooms=2, beds=2, bathrooms=1.0,
    minimum_nights=2, availability_365=200,
    number_of_reviews=25, reviews_per_month=1.5,
    review_scores_rating=4.8, review_scores_cleanliness=4.9,
    review_scores_location=4.7, host_is_superhost=1,
    host_response_rate=95.0, calculated_host_listings_count=1,
    instant_bookable=1, room_type_code=3,
    neighbourhood_freq=0.05, property_type_freq=0.4,
)

print('Logement : T2, 4 personnes, note 4.8, superhost\n')
for name in CITIES:
    m, _, _, _, feats = models[name]['multiple']
    # DataFrame 1 ligne dans l'ordre exact des features du modèle
    X_new = pd.DataFrame([[new_listing.get(f,0) for f in feats]], columns=feats)
    prix  = round(float(m.predict(X_new)[0]), 2)
    print(f'  {name:10s} -> {prix} €/nuit')

## Conclusion

| Ville | R² Simple | R² Multiple | Prix estimé (T2, 4p) |
|-------|-----------|-------------|----------------------|
| Lyon | 0.27 | **0.38** | ~130€/nuit |
| Paris | 0.23 | **0.33** | ~306€/nuit |
| Bordeaux | 0.51 | **0.62** | ~115€/nuit |

**Pistes d'amélioration :** Random Forest, XGBoost, données géographiques, saisonnalité, NLP.